In [ ]:
# Install necessary libraries (you only need to run this once)
%pip install erddapy pandas numpy scikit-learn matplotlib

In [2]:


# Import core data engineering tools
from erddapy import ERDDAP      # Specifically for fetching NOAA data [cite: 16, 42]
import pandas as pd             # For time-series cleaning and data tables [cite: 17, 59]
import numpy as np              # For numerical calculations and weights [cite: 33]
import matplotlib.pyplot as plt # For immediate visual feedback/plotting [cite: 52, 60]

print("Environment setup complete. Ready to fetch NOAA data!")

Environment setup complete. Ready to fetch NOAA data!


In [3]:
pip install ndbc-api

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [41]:
from ndbc_api import NdbcApi

api = NdbcApi()

# PB Pier South: 32.79481789562187N, -117.25924224137366E

# 2. Define your "Virtual Buoy" target location (Example: Monterey Bay area)
# Use 'N/S' and 'E/W' string formatting required by the API
VIRTUAL_LAT = '32.79481789562187N'
VIRTUAL_LON = '-117.25924224137366E'
SEARCH_RADIUS_KM = 100



print(f"Searching for physical NOAA buoys within {SEARCH_RADIUS_KM}km of ({VIRTUAL_LAT}, {VIRTUAL_LON})...")

# 3. Perform a radial search to find neighboring physical stations
nearby_stations = api.radial_search(
    lat=VIRTUAL_LAT, 
    lon=VIRTUAL_LON, 
    radius=SEARCH_RADIUS_KM, 
    units='km'
)

# Display the nearest stations found
nearby_stations.head()

Searching for physical NOAA buoys within 100km of (32.79481789562187N, -117.25924224137366E)...


,Station,Lat,Lon,Elevation,Name,Owner,Program,Type,Includes Meteorology,Includes Currents,Includes Water Quality,DART Program,distance
965,ljac1,32.867,-117.257,9.3,"9410230 - La Jolla, CA",NOS,NOS/CO-OPS,fixed,True,False,False,False,8.037839
966,ljpc1,32.867,-117.257,0.0,"La Jolla, CA (073)",SCRIPPS,IOOS Partners,fixed,True,False,False,False,8.037839
445,46254,32.868,-117.267,0.0,"SCRIPPS Nearshore, CA (201)",SCRIPPS,IOOS Partners,buoy,True,False,False,False,8.178673
1210,sdbc1,32.714,-117.174,NaN,"9410170 - San Diego, CA",NOS,NOS/CO-OPS,fixed,True,False,False,False,12.025736
453,46273,32.930,-117.274,0.0,"Torrey Pines Inner, CA (101)",SCRIPPS,IOOS Partners,buoy,False,False,False,False,15.111216


In [42]:
# Create a dictionary to hold dataframes for each buoy
buoy_data_dict = {}

# Select the top 10 closest stations to check for active telemetry
target_buoys = nearby_stations['Station'].head(10).tolist()

# OPTIMIZED: We only query the 4 modes that realistically contain our features
MODES_TO_TRY = ['stdmet', 'cwind', 'ocean', 'spec']

print(f"Ingesting data using targeted fallback for buoys: {target_buoys}\n")

for station_id in target_buoys:
    data_found = False
    
    for mode in MODES_TO_TRY:
        try:
            data_response = api.get_data(station_id=station_id, mode=mode)
            
            # Format dictionaries directly into structured Pandas tables
            if isinstance(data_response, dict):
                raw_df = pd.DataFrame.from_dict(data_response)
            else:
                raw_df = data_response
                
            if raw_df is None or raw_df.empty:
                continue
                
            # Coerce column names to uppercase to avoid parsing mismatches
            raw_df.columns = [col.upper() for col in raw_df.columns]
            available_cols = raw_df.columns.tolist()
            
            # Extract only the key modeling columns we cares about
            features_to_keep = [col for col in ['WVHT', 'WSPD', 'WTMP', 'DPD', 'APD'] if col in available_cols]
            
            if not features_to_keep:
                continue
                
            # Clean empty timestamps, retaining rows containing at least one metric
            cleaned_df = raw_df[features_to_keep].copy().dropna(how='all')
            
            if not cleaned_df.empty:
                buoy_data_dict[station_id] = cleaned_df
                print(f" ✅ [Success] Station {station_id} parsed via '{mode}'. Rows: {len(cleaned_df)} | Features: {features_to_keep}")
                data_found = True
                break  # Stop cycling modes for this buoy; proceed to the next station
                
        except Exception:
            continue  # Silently drop down to the next mode in the list if an API error occurs
            
    if not data_found:
        print(f" ❌ [No Data] Station {station_id} unavailable across targeted arrays.")

print("\n--- Ingestion Framework Complete ---")
print(f"Active spatial nodes loaded: {list(buoy_data_dict.keys())}")

Ingesting data using targeted fallback for buoys: ['ljac1', 'ljpc1', '46254', 'sdbc1', '46273', '46266', '46225', '46258', 'npqc1', '46235']

 ✅ [Success] Station ljac1 parsed via 'stdmet'. Rows: 6868 | Features: ['WVHT', 'WSPD', 'WTMP', 'DPD', 'APD']
 ✅ [Success] Station ljpc1 parsed via 'stdmet'. Rows: 719 | Features: ['WVHT', 'WSPD', 'WTMP', 'DPD', 'APD']
 ✅ [Success] Station 46254 parsed via 'stdmet'. Rows: 1438 | Features: ['WVHT', 'WSPD', 'WTMP', 'DPD', 'APD']
 ✅ [Success] Station sdbc1 parsed via 'stdmet'. Rows: 6242 | Features: ['WVHT', 'WSPD', 'WTMP', 'DPD', 'APD']
 ❌ [No Data] Station 46273 unavailable across targeted arrays.
 ✅ [Success] Station 46266 parsed via 'stdmet'. Rows: 1437 | Features: ['WVHT', 'WSPD', 'WTMP', 'DPD', 'APD']
 ✅ [Success] Station 46225 parsed via 'stdmet'. Rows: 1439 | Features: ['WVHT', 'WSPD', 'WTMP', 'DPD', 'APD']
 ✅ [Success] Station 46258 parsed via 'stdmet'. Rows: 1439 | Features: ['WVHT', 'WSPD', 'WTMP', 'DPD', 'APD']
 ❌ [No Data] Station npqc1

In [43]:
import os

# 1. Create a local 'data' directory if it doesn't exist
os.makedirs("../data", exist_ok=True)

print("Exporting cleaned time-series tables to local storage...")

# 2. Iterate through your successful nodes and save them as separate CSV files
for station_id, dataframe in buoy_data_dict.items():
    output_path = f"../data/{station_id}_cleaned.csv"
    dataframe.to_csv(output_path)
    print(f" Saved data for station {station_id} -> {output_path} ({len(dataframe)} rows)")

print("\n🎉 Phase I: Data Ingestion Notebook Complete!")

Exporting cleaned time-series tables to local storage...
 Saved data for station ljac1 -> ../data/ljac1_cleaned.csv (6868 rows)
 Saved data for station ljpc1 -> ../data/ljpc1_cleaned.csv (719 rows)
 Saved data for station 46254 -> ../data/46254_cleaned.csv (1438 rows)
 Saved data for station sdbc1 -> ../data/sdbc1_cleaned.csv (6242 rows)
 Saved data for station 46266 -> ../data/46266_cleaned.csv (1437 rows)
 Saved data for station 46225 -> ../data/46225_cleaned.csv (1439 rows)
 Saved data for station 46258 -> ../data/46258_cleaned.csv (1439 rows)

🎉 Phase I: Data Ingestion Notebook Complete!


In [44]:
import os

# Create a local 'data' directory if it doesn't exist
os.makedirs("../data", exist_ok=True)

print("Exporting cleaned time-series tables to local storage...")

# Iterate through your successful nodes and save them as separate CSV files
for station_id, dataframe in buoy_data_dict.items():
    output_path = f"../data/{station_id}_cleaned.csv"
    dataframe.to_csv(output_path)
    print(f" Saved data for station {station_id} -> {output_path} ({len(dataframe)} rows)")

print("\n🎉 Phase I: Data Ingestion Notebook Complete!")

Exporting cleaned time-series tables to local storage...
 Saved data for station ljac1 -> ../data/ljac1_cleaned.csv (6868 rows)
 Saved data for station ljpc1 -> ../data/ljpc1_cleaned.csv (719 rows)
 Saved data for station 46254 -> ../data/46254_cleaned.csv (1438 rows)
 Saved data for station sdbc1 -> ../data/sdbc1_cleaned.csv (6242 rows)
 Saved data for station 46266 -> ../data/46266_cleaned.csv (1437 rows)
 Saved data for station 46225 -> ../data/46225_cleaned.csv (1439 rows)
 Saved data for station 46258 -> ../data/46258_cleaned.csv (1439 rows)

🎉 Phase I: Data Ingestion Notebook Complete!
